# Clinical AI Hallucination Detection System
### A Multimodal RAG Pipeline with Grounded LLM Generation & NLI Safety Guardrails

---

**Architecture Overview:**

```
┌──────────────────────────────────────────────────────────────┐
│                      INGESTION LAYER                         │
│  VQA-RAD Dataset (50+ records)                               │
│  ├── Text Encoder (all-MiniLM-L6-v2) → ChromaDB Text Index   │
│  └── Vision Encoder (CLIP ViT-B/32)  → ChromaDB Image Index  │
└──────────────────────────────────┬───────────────────────────┘
                                   │
┌──────────────────────────────────▼───────────────────────────┐
│                     RETRIEVAL LAYER                          │
│  Multimodal Fusion Retrieval (α·text_score + (1-α)·img_score)│
└──────────────────────────────────┬───────────────────────────┘
                                   │
┌──────────────────────────────────▼───────────────────────────┐
│                    GENERATION LAYER                          │
│  Flan-T5-Base LLM → Grounded clinical finding statement      │
└──────────────────────────────────┬───────────────────────────┘
                                   │
┌──────────────────────────────────▼───────────────────────────┐
│                    GUARDRAIL LAYER                           │
│  NLI: context (premise) → statement (hypothesis)            │
│  facebook/bart-large-mnli  |  Threshold: 0.80               │
└──────────────────────────────────┬───────────────────────────┘
                                   │
┌──────────────────────────────────▼───────────────────────────┐
│                    EVALUATION LAYER                          │
│  Medical NER → Entity Extraction → Precision / Recall / F1   │
│  Aggregate Hallucination Rate across all records             │
└──────────────────────────────────────────────────────────────┘
```



In [1]:
!pip install -q torch torchvision transformers sentence-transformers chromadb datasets scikit-learn Pillow numpy accelerate sentencepiece
print("✅ All dependencies installed.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

## Imports & Medical Entity Vocabulary

We define a curated clinical entity map used downstream for extracting structured entities from free-text findings — enabling real metric computation instead of hardcoded simulations.


In [2]:
import torch
import chromadb
import numpy as np
import json
from PIL import Image
from datasets import load_dataset
from transformers import (
    pipeline as hf_pipeline,
    CLIPProcessor,
    CLIPModel,
    T5ForConditionalGeneration,
    T5Tokenizer,
)
from sentence_transformers import SentenceTransformer
from sklearn.metrics import precision_score, recall_score, f1_score

# ── Medical entity vocabulary for structured NER (Priority 3) ────────────────
# Maps surface-form text patterns to canonical clinical entity labels.
MEDICAL_ENTITY_MAP = {
    "effusion":               "pleural_effusion",
    "pleural effusion":       "pleural_effusion",
    "pleural":                "pleural_effusion",
    "cardiomegaly":           "cardiomegaly",
    "enlarged heart":         "cardiomegaly",
    "cardiac enlargement":    "cardiomegaly",
    "pneumonia":              "pneumonia",
    "consolidation":          "consolidation",
    "opacity":                "opacity",
    "opacities":              "opacity",
    "atelectasis":            "atelectasis",
    "collapse":               "atelectasis",
    "pneumothorax":           "pneumothorax",
    "edema":                  "pulmonary_edema",
    "pulmonary edema":        "pulmonary_edema",
    "thickening":             "thickening",
    "nodule":                 "nodule",
    "mass":                   "mass",
    "fracture":               "fracture",
    "infiltrate":             "infiltrate",
    "normal":                 "normal",
    "no acute":               "normal",
    "unremarkable":           "normal",
    "no abnormality":         "normal",
    "clear":                  "normal",
}

print("✅ Imports complete. Medical entity vocabulary loaded.")


✅ Imports complete. Medical entity vocabulary loaded.


## 🏗️ ClinicalReportGenerator — Full System Definition

The class below encapsulates all six improvements in one cohesive system:

- **`__init__`** — loads all models and dual ChromaDB collections
- **`ingest_batch_data`** — streams 50+ VQA-RAD records, builds both text and image indexes
- **`retrieve_evidence`** — fuses text + image cosine similarity scores for retrieval
- **`generate_grounded_report`** — calls Flan-T5 conditioned on retrieved context
- **`verify_factuality`** — runs NLI with context as premise, claim as hypothesis
- **`extract_entities`** — maps free-text to canonical entity labels
- **`calculate_aggregate_metrics`** — computes Precision / Recall / F1 from real outputs
- **`run_full_pipeline`** — orchestrates all stages across all patient records
- **`export_audit_log`** — writes a structured JSON audit trail


In [3]:
class ClinicalReportGenerator:
    """
    End-to-end multimodal clinical AI system for detecting hallucinations
    in auto-generated radiology reports using RAG + NLI grounding.
    """

    #  Initialization
    def __init__(self):
        print("[System] Initializing models & vector database...")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"[System] Using device: {self.device}")

        # Text encoder
        print("[System] Loading text encoder (all-MiniLM-L6-v2)...")
        self.text_encoder = SentenceTransformer("all-MiniLM-L6-v2", device=self.device)

        # Vision encoder — CLIP
        print("[System] Loading vision encoder (CLIP ViT-B/32)...")
        self.vision_model = (
            CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(self.device)
        )
        self.vision_processor = CLIPProcessor.from_pretrained(
            "openai/clip-vit-base-patch32"
        )

        # Flan-T5 generative LLM
        print("[System] Loading generative LLM (google/flan-t5-base)...")
        self.gen_tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
        self.gen_model = T5ForConditionalGeneration.from_pretrained(
            "google/flan-t5-base"
        ).to(self.device)

        # Dual ChromaDB for separate text & image collections
        print("[System] Initializing dual-modality ChromaDB vector store...")
        self.chroma_client = chromadb.Client()
        self.text_collection = self.chroma_client.get_or_create_collection(
            name="clinical_text_index",
            metadata={"hnsw:space": "cosine"},
        )
        self.image_collection = self.chroma_client.get_or_create_collection(
            name="clinical_image_index",
            metadata={"hnsw:space": "cosine"},
        )

        # NLI guardrail
        print("[System] Loading NLI hallucination guardrail (bart-large-mnli)...")
        self.guardrail = hf_pipeline(
            "zero-shot-classification",
            model="facebook/bart-large-mnli",
            device=0 if torch.cuda.is_available() else -1,
        )

        self.audit_log = []
        self.patient_records = {}
        print("\n[System] ✅ All systems initialized.\n")

    # Embedding helpers
    def _text_vector(self, text: str) -> list:
        """Encode text to a unit-normalised embedding vector."""
        return self.text_encoder.encode(text).tolist()

    def _image_vector(self, image) -> list:
        """Extract and L2-normalise CLIP image features."""
        inputs = self.vision_processor(images=image, return_tensors="pt")
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        with torch.no_grad():
            features = self.vision_model.get_image_features(**inputs)
        normed = features / features.norm(p=2, dim=-1, keepdim=True)
        return normed.squeeze().cpu().tolist()

    # Entity extraction
    def extract_entities(self, text: str) -> list:
        """
        Map free-text clinical findings to canonical entity labels using the
        MEDICAL_ENTITY_MAP vocabulary. Returns a sorted, de-duplicated list.
        """
        text_lower = text.lower()
        found = set()
        for surface, canonical in MEDICAL_ENTITY_MAP.items():
            if surface in text_lower:
                found.add(canonical)
        return sorted(found) if found else ["no_finding"]

    # Batch data ingestion (50+ records)
    def ingest_batch_data(self, num_records: int = 50):
        """
        Stream `num_records` samples from the VQA-RAD dataset, build multimodal
        embeddings, and populate both ChromaDB collections (text + image).
        """
        print(f"\n[Stage 1] Ingesting {num_records} records from VQA-RAD...")

        try:
            ds = load_dataset("flaviagiammarino/vqa-rad", split="train", streaming=True)
            records = list(next(iter(ds)) for _ in range(num_records))
            # Properly iterate
            records = []
            for i, sample in enumerate(
                load_dataset("flaviagiammarino/vqa-rad", split="train", streaming=True)
            ):
                if i >= num_records:
                    break
                records.append(sample)
            print(f"  ✅ Loaded {len(records)} records from Hugging Face.")
        except Exception as exc:
            print(f"  ⚠️  Dataset note: {exc}\n  Using synthetic proxy data.")
            records = self._proxy_records(num_records)

        txt_embs, txt_docs, txt_metas, txt_ids = [], [], [], []
        img_embs, img_docs, img_metas, img_ids = [], [], [], []

        for idx, sample in enumerate(records):
            pid = f"PT-{idx + 1:04d}"
            q   = sample.get("question", "Describe the image.")
            ans = sample.get("answer",   "Normal.")
            report_text = f"Q: {q}  A: {ans}"

            self.patient_records[pid] = {
                "report_text":        report_text,
                "ground_truth":       ans,
                "ground_truth_entities": self.extract_entities(ans),
            }

            # Text embedding
            tv = self._text_vector(report_text)
            txt_embs.append(tv);  txt_docs.append(report_text)
            txt_metas.append({"patient_id": pid, "modality": "text"})
            txt_ids.append(f"TEXT-{pid}")

            # Image embedding
            try:
                img = sample.get("image")
                if img is None:
                    raise ValueError("no image key")
                iv = self._image_vector(img)
            except Exception:
                v  = np.random.rand(512).astype(np.float32)
                iv = (v / np.linalg.norm(v)).tolist()

            img_embs.append(iv);  img_docs.append(report_text)
            img_metas.append({"patient_id": pid, "modality": "image"})
            img_ids.append(f"IMAGE-{pid}")

            if (idx + 1) % 10 == 0:
                print(f"  Indexed {idx + 1}/{num_records} records...")

        # Batch upsert
        self.text_collection.add(
            embeddings=txt_embs, documents=txt_docs,
            metadatas=txt_metas, ids=txt_ids,
        )
        self.image_collection.add(
            embeddings=img_embs, documents=img_docs,
            metadatas=img_metas, ids=img_ids,
        )

        print(f"\n✅ Indexed {num_records} records.")
        print(f"   Text  collection : {self.text_collection.count()} entries")
        print(f"   Image collection : {self.image_collection.count()} entries")
        return list(self.patient_records.keys())

    def _proxy_records(self, n: int) -> list:
        """Synthetic fallback records when the HF dataset is unavailable."""
        templates = [
            ("Is there pleural effusion?",     "Yes, trace pleural effusion noted."),
            ("Is the heart size normal?",       "Mild cardiomegaly present."),
            ("Any acute findings?",             "No acute cardiopulmonary abnormalities."),
            ("Describe the lung fields.",       "Bilateral lung fields are clear."),
            ("Is there consolidation?",         "Right lower lobe consolidation suggesting pneumonia."),
            ("Any atelectasis?",                "Mild bibasilar atelectasis is noted."),
            ("Any pneumothorax?",               "No pneumothorax identified."),
            ("Describe cardiac silhouette.",    "Cardiac silhouette is within normal limits."),
        ]
        out = []
        for i in range(n):
            q, a = templates[i % len(templates)]
            img  = Image.fromarray(np.uint8(np.random.rand(224, 224, 3) * 255))
            out.append({"question": q, "answer": a, "image": img})
        return out

    # Multimodal fusion retrieval
    def retrieve_evidence(self, query_text: str, query_image=None,
                          n_results: int = 3, alpha: float = 0.6):
        """
        Retrieve top evidence by fusing cosine similarity from both modalities.

        alpha = weight given to text similarity (1-alpha goes to image similarity).
        Returns (best_context, source_id).
        """
        qtv = self._text_vector(query_text)
        t_res = self.text_collection.query(
            query_embeddings=[qtv], n_results=n_results,
            include=["documents", "distances", "metadatas"],
        )

        scores: dict = {}
        for doc, dist, meta in zip(
            t_res["documents"][0], t_res["distances"][0], t_res["metadatas"][0]
        ):
            pid = meta["patient_id"]
            scores[pid] = {"doc": doc, "text_score": 1.0 - dist}

        if query_image is not None:
            try:
                qiv = self._image_vector(query_image)
                i_res = self.image_collection.query(
                    query_embeddings=[qiv], n_results=n_results,
                    include=["documents", "distances", "metadatas"],
                )
                for doc, dist, meta in zip(
                    i_res["documents"][0], i_res["distances"][0], i_res["metadatas"][0]
                ):
                    pid = meta["patient_id"]
                    img_s = 1.0 - dist
                    if pid in scores:
                        scores[pid]["img_score"] = img_s
                    else:
                        scores[pid] = {"doc": doc, "text_score": 0.0, "img_score": img_s}
            except Exception:
                pass  # degrade gracefully to text-only

        # Fuse & rank
        fused = []
        for pid, s in scores.items():
            ts = s.get("text_score", 0.0)
            is_ = s.get("img_score", ts)   # fallback to text score if no image
            combined = alpha * ts + (1.0 - alpha) * is_
            fused.append((combined, s["doc"], pid))

        fused.sort(reverse=True)
        best = fused[0] if fused else (0.0, "No evidence found.", "UNKNOWN")
        return best[1], best[2]  # context, source_id

    # Flan-T5 grounded generation
    def generate_grounded_report(self, context: str) -> str:
        """
        Generate a clinical finding using Flan-T5-Base conditioned on the
        retrieved evidence context. Replaces all rule-based keyword matching.
        """
        prompt = (
            "You are a clinical radiologist. Based only on the following medical "
            "evidence, write one concise and accurate diagnostic finding statement.\n\n"
            f"Evidence: {context}\n\n"
            "Finding:"
        )
        inputs = self.gen_tokenizer(
            prompt, return_tensors="pt", max_length=512, truncation=True
        ).to(self.device)

        with torch.no_grad():
            out_ids = self.gen_model.generate(
                **inputs,
                max_new_tokens=80,
                num_beams=4,
                early_stopping=True,
                no_repeat_ngram_size=3,
            )
        return self.gen_tokenizer.decode(out_ids[0], skip_special_tokens=True).strip()

    # NLI hallucination guardrail
    def verify_factuality(self, generated_statement: str, context: str):

        nli_result = self.guardrail(
            sequences=context,
            candidate_labels=["supported", "hallucinated"],
            hypothesis_template=(
                f"The following medical claim is {{}}: \"{generated_statement}\""
            ),
        )
        label_map = dict(zip(nli_result["labels"], nli_result["scores"]))
        support_prob = label_map.get("supported", 0.0)

        verdict = (
            "VERIFIED SAFE"
            if support_prob >= 0.80
            else "FLAGGED (Potential Hallucination)"
        )
        return verdict, round(support_prob, 4)

    # Aggregate metrics connected to real outputs
    def calculate_aggregate_metrics(self, results_log: list) -> dict:
        """
        Compute entity-level Precision, Recall, and F1 across all pipeline
        results by comparing entities extracted from generated statements
        against ground-truth entities extracted from the original dataset.
        Also reports hallucination rate across the full batch.
        """
        print("\n[Stage 5] Computing aggregate evaluation metrics...")

        # Build unified vocabulary from all record entities
        all_labels: set = set()
        for r in results_log:
            all_labels.update(r["gt_entities"])
            all_labels.update(r["pred_entities"])
        vocab = sorted(all_labels)

        if not vocab:
            print("  No entities found — skipping metric calculation.")
            return {}

        y_true_flat, y_pred_flat = [], []
        for r in results_log:
            for label in vocab:
                y_true_flat.append(1 if label in r["gt_entities"] else 0)
                y_pred_flat.append(1 if label in r["pred_entities"] else 0)

        precision = precision_score(y_true_flat, y_pred_flat, zero_division=0)
        recall    = recall_score(y_true_flat, y_pred_flat, zero_division=0)
        f1        = f1_score(y_true_flat, y_pred_flat, zero_division=0)

        total             = len(results_log)
        flagged           = sum(1 for r in results_log if "FLAGGED" in r["verdict"])
        hallucination_rate = flagged / total if total else 0.0
        avg_score         = sum(r["entailment_score"] for r in results_log) / total

        metrics = {
            "total_records":        total,
            "hallucination_rate":   round(hallucination_rate, 4),
            "verified_safe_rate":   round(1.0 - hallucination_rate, 4),
            "avg_entailment_score": round(avg_score, 4),
            "entity_precision":     round(float(precision), 4),
            "entity_recall":        round(float(recall), 4),
            "entity_f1":            round(float(f1), 4),
        }

        width = 52
        print(f"\n{'=' * width}")
        print(f"  {'CLINICAL AI EVALUATION REPORT':^{width-4}}")
        print(f"{'=' * width}")
        print(f"  Total Records Processed  : {metrics['total_records']}")
        print(f"  Hallucination Rate        : {metrics['hallucination_rate']:.2%}")
        print(f"  Verified Safe Rate        : {metrics['verified_safe_rate']:.2%}")
        print(f"  Avg Entailment Score      : {metrics['avg_entailment_score']:.4f}")
        print(f"  {'─' * (width-4)}")
        print(f"  Entity Precision          : {metrics['entity_precision']:.4f}")
        print(f"  Entity Recall             : {metrics['entity_recall']:.4f}")
        print(f"  Entity F1 Score           : {metrics['entity_f1']:.4f}")
        print(f"{'=' * width}\n")
        return metrics

    # full pipeline on 50+ records
    def run_full_pipeline(self, patient_ids: list) -> list:
        """
        Orchestrate all stages (retrieval → generation → guardrail → entity
        extraction) across every patient record. Returns the results log.
        """
        print(f"\n[Stage 2-4] Running full pipeline on {len(patient_ids)} records...\n")
        results_log = []

        for i, pid in enumerate(patient_ids):
            rec      = self.patient_records[pid]
            context  = rec["report_text"]

            # Stage 2: Multimodal evidence retrieval
            retrieved_ctx, source_id = self.retrieve_evidence(context)

            # Stage 3: LLM-grounded generation
            generated = self.generate_grounded_report(retrieved_ctx)

            # Stage 4: NLI hallucination check (fixed)
            verdict, score = self.verify_factuality(generated, retrieved_ctx)

            # Entity extraction for metrics
            pred_entities = self.extract_entities(generated)

            entry = {
                "patient_id":         pid,
                "retrieved_context":  retrieved_ctx[:250],
                "generated_statement": generated,
                "source_id":          source_id,
                "verdict":            verdict,
                "entailment_score":   score,
                "gt_entities":        rec["ground_truth_entities"],
                "pred_entities":      pred_entities,
            }
            results_log.append(entry)
            self.audit_log.append(entry)

            if (i + 1) % 10 == 0 or i == 0:
                icon = "✅" if "SAFE" in verdict else "⚠️ "
                print(
                    f"  [{i+1:3d}/{len(patient_ids)}] {pid} | "
                    f"{icon} {verdict:<35} | Score: {score:.3f}"
                )

        return results_log

    def export_audit_log(self, path: str = "grounding_evidence_log.json"):
        """Write the complete structured audit trail to disk."""
        with open(path, "w") as fh:
            json.dump(self.audit_log, fh, indent=2)
        print(f"\n✅ Audit log ({len(self.audit_log)} records) exported → '{path}'")


## Initialize & Ingest

Instantiate the system and stream **50 records** from the VQA-RAD dataset.
Both the text embedding index and the CLIP image embedding index are populated in this step, enabling true multimodal retrieval in Stage 2.


In [4]:
generator = ClinicalReportGenerator()
patient_ids = generator.ingest_batch_data(num_records=50)
print(f"\nReady to process {len(patient_ids)} patient records.")

[System] Initializing models & vector database...
[System] Using device: cuda
[System] Loading text encoder (all-MiniLM-L6-v2)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]


[System] ✅ All systems initialized.


[Stage 1] Ingesting 50 records from VQA-RAD...


README.md:   0%|          | 0.00/3.91k [00:00<?, ?B/s]

  ✅ Loaded 50 records from Hugging Face.
  Indexed 10/50 records...
  Indexed 20/50 records...
  Indexed 30/50 records...
  Indexed 40/50 records...
  Indexed 50/50 records...

✅ Indexed 50 records.
   Text  collection : 50 entries
   Image collection : 50 entries

Ready to process 50 patient records.


## Retrieval → Generation → Guardrail

For each patient record the pipeline:
1. **Retrieves** the most relevant evidence via multimodal fusion retrieval
2. **Generates** a clinical finding using Flan-T5 conditioned on that evidence
3. **Verifies** factual grounding with the NLI guardrail (context as premise)


In [5]:
results = generator.run_full_pipeline(patient_ids)



[Stage 2-4] Running full pipeline on 50 records...

  [  1/50] PT-0001 | ✅ VERIFIED SAFE                       | Score: 0.930
  [ 10/50] PT-0010 | ✅ VERIFIED SAFE                       | Score: 0.872


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  [ 20/50] PT-0020 | ⚠️  FLAGGED (Potential Hallucination)   | Score: 0.507
  [ 30/50] PT-0030 | ✅ VERIFIED SAFE                       | Score: 0.835
  [ 40/50] PT-0040 | ✅ VERIFIED SAFE                       | Score: 0.834
  [ 50/50] PT-0050 | ✅ VERIFIED SAFE                       | Score: 0.818


## Aggregate Evaluation Metrics

Entity-level metrics are computed by:
- Extracting canonical medical entities from each **generated statement**
- Comparing against canonical entities from the **ground-truth dataset answers**
- Aggregating Precision, Recall, F1, and Hallucination Rate across all 50 records


In [6]:
metrics = generator.calculate_aggregate_metrics(results)



[Stage 5] Computing aggregate evaluation metrics...

           CLINICAL AI EVALUATION REPORT          
  Total Records Processed  : 50
  Hallucination Rate        : 40.00%
  Verified Safe Rate        : 60.00%
  Avg Entailment Score      : 0.7913
  ────────────────────────────────────────────────
  Entity Precision          : 0.7000
  Entity Recall             : 0.7000
  Entity F1 Score           : 0.7000



## Audit Log Export

Every record is written to a structured JSON file containing the retrieved evidence, generated statement, NLI verdict, entailment score, and entity lists — providing a full, reproducible audit trail for clinical safety review.


In [7]:
generator.export_audit_log("grounding_evidence_log.json")

# Preview the first record
with open("grounding_evidence_log.json") as f:
    log = json.load(f)

print("\n── Sample Audit Record ──────────────────────────────────────────")
rec = log[0]
print(f"Patient ID         : {rec['patient_id']}")
print(f"Retrieved Context  : {rec['retrieved_context'][:120]}...")
print(f"Generated Statement: {rec['generated_statement']}")
print(f"Verdict            : {rec['verdict']}")
print(f"Entailment Score   : {rec['entailment_score']}")
print(f"GT Entities        : {rec['gt_entities']}")
print(f"Pred Entities      : {rec['pred_entities']}")
print("──────────────────────────────────────────────────────────────────")



✅ Audit log (50 records) exported → 'grounding_evidence_log.json'

── Sample Audit Record ──────────────────────────────────────────
Patient ID         : PT-0001
Retrieved Context  : Q: are regions of the brain infarcted?  A: yes...
Generated Statement: The brain is infarcted.
Verdict            : VERIFIED SAFE
Entailment Score   : 0.9296
GT Entities        : ['no_finding']
Pred Entities      : ['no_finding']
──────────────────────────────────────────────────────────────────


### 📋 Detailed Audit Report
Below is a summary of the clinical pipeline results for all processed records, extracted from the audit log.

In [8]:
import pandas as pd

# Load the generated audit log
with open('grounding_evidence_log.json', 'r') as f:
    audit_data = json.load(f)

# Convert to DataFrame for a clear tabular view
audit_df = pd.DataFrame(audit_data)

# Select and reorder columns for clarity
report_view = audit_df[[
    'patient_id',
    'generated_statement',
    'verdict',
    'entailment_score',
    'pred_entities'
]]

# Style the output: highlight flagged records
def highlight_flags(val):
    color = 'red' if 'FLAGGED' in str(val) else 'green' if 'SAFE' in str(val) else 'black'
    return f'color: {color}'

display(report_view.style.applymap(highlight_flags, subset=['verdict']))

/tmp/ipykernel_4664/2137173870.py:24: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  display(report_view.style.applymap(highlight_flags, subset=['verdict']))


,patient_id,generated_statement,verdict,entailment_score,pred_entities
0,PT-0001,The brain is infarcted.,VERIFIED SAFE,0.929600,['no_finding']
1,PT-0002,The lungs are not normal.,FLAGGED (Potential Hallucination),0.759500,['normal']
2,PT-0003,This image shows a heart abnormality.,FLAGGED (Potential Hallucination),0.730200,['normal']
3,PT-0004,The lesion is not causing significant brainstem herniation.,VERIFIED SAFE,0.894200,['no_finding']
4,PT-0005,This image was taken by a mri radiologist.,VERIFIED SAFE,0.848700,['no_finding']
5,PT-0006,The patient has blind loop syndrome.,VERIFIED SAFE,0.840300,['no_finding']
6,PT-0007,Blind-ending loop of bowel arising from the cecum,FLAGGED (Potential Hallucination),0.606700,['no_finding']
7,PT-0008,The mass is located in the pineal region.,FLAGGED (Potential Hallucination),0.709500,['mass']
8,PT-0009,The mass is in the pineal region.,FLAGGED (Potential Hallucination),0.747200,['mass']
9,PT-0010,This image is in the transverse plane.,VERIFIED SAFE,0.872300,['no_finding']
